# 4장 실습: Recommendations 데이터로 Cypher 실습

실습 가이드 4장 4-3의 코드를 순서대로 실행하는 노트북입니다. 설명은 실습 가이드를 함께 봅니다.
도커를 실행하고 .env 를 수정합니다.

| 항목 | 내용 |
|---|---|
| 연결할 DB | Recommendations (샌드박스, 강사 AuraDB, 또는 Docker) |
| .env 설정 | Recommendations DB의 NEO4J 네 줄 |

.env를 바꿨다면 노트북 상단의 `Restart`로 커널을 재시작한 뒤 처음부터 실행합니다. 연결 셀의 `connect_graph()`는 .env를 다시 읽지만, 앞 셀에서 만든 객체가 이전 DB를 가리킬 수 있으므로 처음부터 다시 실행합니다.

4-1, 4-2의 기본 조회는 Neo4j Browser에서 실습합니다. 이 노트북은 4-3과 실습 과제를 Python으로 실행하는 부분입니다.

## 연결 확인

`Neo4jGraph`는 연결할 때 DB 스키마를 읽습니다. 이때 APOC 플러그인을 쓰므로, APOC가 없는 DB에서는 `Could not use APOC procedures` 오류가 납니다(Sandbox, AuraDB, 실습용 Docker 이미지에는 들어 있습니다).

출력된 레이블별 노드 수로 이 차시에 맞는 DB에 연결됐는지 확인합니다.

In [1]:
# 이대로 해도 되지만 graphrag_tools.py 를 사용한다.
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

load_dotenv()

# .env의 접속 정보를 한 묶음으로 둔다
NEO4J = dict(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE"),
)
graph = Neo4jGraph(**NEO4J)

print(graph.query("RETURN 'connected' AS status"))


[{'status': 'connected'}]


In [5]:
print(graph.schema)   # 노드 속성, 관계 속성, 관계 패턴을 문자열로 반환

Node properties:
Movie {url: STRING, runtime: INTEGER, revenue: INTEGER, imdbRating: FLOAT, released: STRING, countries: LIST, languages: LIST, plot: STRING, imdbVotes: INTEGER, imdbId: STRING, year: INTEGER, poster: STRING, movieId: STRING, tmdbId: STRING, title: STRING, embedding: LIST, budget: INTEGER}
Genre {name: STRING}
User {userId: STRING, name: STRING}
Actor {url: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, tmdbId: STRING, bornIn: STRING, bio: STRING, poster: STRING}
Director {url: STRING, bornIn: STRING, born: DATE, died: DATE, tmdbId: STRING, imdbId: STRING, name: STRING, poster: STRING, bio: STRING}
Person {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}
Document {text: STRING, embedding: LIST, id: STRING}
Country {id: STRING}
Field {id: STRING}
Award {id: STRING}
Organization {id: STRING}
Relationship properties:
RATED {rating: FLOAT, timestamp: INTEGER}
ACTED_IN {role: STRING

In [2]:
from tools.graphrag_tools import connect_graph

# .env의 접속 정보로 연결하고 레이블별 노드 수를 출력한다. expect의 레이블이 없으면 다른 DB에 연결됐다고 알려 준다.
# NEO4J는 접속 정보 묶음이다. 다른 LangChain 객체를 만들 때 **NEO4J로 그대로 넘긴다(실습 가이드 6-2).
# 연결 오류는 이 줄에서 난다. ServiceUnavailable: 서버가 꺼져 있거나 시작 중 / AuthError: 비밀번호 / APOC 오류: 플러그인 없음
graph, NEO4J = connect_graph(expect=["Movie", "User", "Genre"])

연결: bolt://localhost:7687 | DB: neo4j
{'label': 'Actor', 'n': 15443}
{'label': 'Movie', 'n': 9125}
{'label': 'Director', 'n': 3604}
{'label': 'User', 'n': 671}
{'label': 'Genre', 'n': 20}


In [11]:
# 샌드박스에서 사용한 쿼리를 실행해 본다. 결과가 나오면 연결 성공이다.
# 실습 가이드 6-3의 쿼리

rows = graph.query("""
MATCH (m:Movie)
WHERE m.year >= 2000
RETURN m.title AS title, m.year AS year
ORDER BY m.year DESC
LIMIT 5
""")
for row in rows:
    print(row["title"], row["year"])

Shin Godzilla 2016
Mohenjo Daro 2016
Rustom 2016
Sharknado 4: The 4th Awakens 2016
Hell or High Water 2016


## 1. 파라미터 쿼리 (실습 가이드 4-3)

값을 쿼리 문자열에 직접 붙이지 않고 `$파라미터`로 넘깁니다. 쿼리를 재사용하기 쉽고, 입력값으로 쿼리 구조를 바꾸는 주입 공격을 막습니다.

아래 쿼리는 영화에서 사용자로, 사용자에서 다른 영화로 이동하는 2-hop 추천입니다.

In [3]:
# $movie는 파라미터 자리표시자다. 값은 params로 따로 넘기고 문자열을 이어 붙이지 않는다.
# (따옴표 처리 실수와 Cypher 인젝션을 막고, 같은 쿼리를 값만 바꿔 재사용할 수 있다)
# 패턴 읽기: Crimson Tide를 평가한 사용자(u)가 평가한 다른 영화(rec) = 가장 단순한 협업 필터링
recommend_query = """
MATCH (m:Movie {title: $movie})<-[:RATED]-(u:User)-[:RATED]->(rec:Movie)
RETURN DISTINCT rec.title AS recommendation
LIMIT 20
"""
for row in graph.query(recommend_query, params={"movie": "Crimson Tide"}):
    print(row["recommendation"])

Mission: Impossible
Fargo
Pretty Woman
Beauty and the Beast
Silence of the Lambs, The
Batman
Dances with Wolves
Terminator 2: Judgment Day
Aladdin
Ghost
Home Alone
True Romance
Nightmare Before Christmas, The
Sleepless in Seattle
Secret Garden, The
Schindler's List
Robin Hood: Men in Tights
Piano, The
Philadelphia
Mrs. Doubtfire


## 실습 과제

위의 셀을 모두 실행한 뒤 진행합니다. 과제마다 문제, 정답 코드, 해설 순서로 되어 있습니다. 먼저 직접 작성해 본 뒤 정답 코드와 비교합니다.

### 과제 1. 별점 4.0 이상을 준 사용자만 대상

1번 셀의 추천 쿼리를 고쳐, Crimson Tide에 별점 4.0 이상을 준 사용자만 대상으로 추천하세요. 관계에 변수를 붙이고(`-[r1:RATED]-`) `WHERE`로 조건을 겁니다. 전체 평가자 수와 4.0 이상 평가자 수도 함께 확인합니다.

In [8]:
# 과제 1 정답
# 관계에 변수(r1)를 붙이면 관계 속성(rating)으로 거를 수 있다. 조건은 기준 영화에 준 별점에만 건다.
high_query = """
MATCH (m:Movie {title: $movie})<-[r1:RATED]-(u:User)-[:RATED]->(rec:Movie)
WHERE r1.rating >= 4.0
RETURN DISTINCT rec.title AS recommendation
LIMIT 20
"""
for row in graph.query(high_query, params={"movie": "Crimson Tide"}):
    print(row["recommendation"])

# 얼마나 걸러졌는지 보여 주기 위해 전체 사용자와 4.0 이상 사용자를 한 번에 센다.
# count(CASE ... END)는 조건이 거짓이면 null이 되어 세지 않는 점을 이용한다.
print(graph.query("""
MATCH (m:Movie {title: $movie})<-[r:RATED]-(u:User)
RETURN count(u) AS all_users, count(CASE WHEN r.rating >= 4.0 THEN 1 END) AS high_users
""", params={"movie": "Crimson Tide"}))

Mission: Impossible
Fargo
Pretty Woman
Beauty and the Beast
Silence of the Lambs, The
Batman
Dances with Wolves
Terminator 2: Judgment Day
Aladdin
Ghost
Home Alone
True Romance
Nightmare Before Christmas, The
Sleepless in Seattle
Secret Garden, The
Schindler's List
Robin Hood: Men in Tights
Piano, The
Philadelphia
Mrs. Doubtfire
[{'all_users': 80, 'high_users': 56}]


#### 과제 1 해설

실행 결과 (2026-11-01): Crimson Tide를 평가한 사용자 80명 중 4.0 이상을 준 사용자는 56명입니다. 그런데 추천 목록 20편은 1번 셀(조건 없음)과 똑같이 나왔습니다.

- 쿼리가 틀린 것이 아닙니다. `ORDER BY` 없이 `LIMIT 20`만 쓰면 DB가 먼저 찾은 20편을 돌려주는데, 4.0 이상을 준 56명만으로도 같은 영화들이 먼저 찾아졌기 때문입니다. 조건의 효과를 보려면 과제 2처럼 정렬하거나 개수를 세야 합니다.
- 관계에 변수(`r1`)를 붙여야 관계의 속성(`rating`)으로 조건을 걸 수 있습니다.
- 확장: 추천할 영화도 높게 평가한 경우만 남기려면 두 번째 관계에도 변수를 붙이고(`-[r2:RATED]->`) `AND r2.rating >= 4.0`을 더합니다.

### 과제 2. 겹친 사용자 수가 많은 순서로 정렬

추천 결과를 "Crimson Tide와 그 영화를 함께 평가한 사용자 수"가 많은 순서로 정렬하세요. `count(u)`를 씁니다.

In [9]:
# 과제 2 정답
# DISTINCT 대신 count(u)로 집계하면 rec별로 겹친 사용자 수가 나온다.
# 집계 함수가 아닌 RETURN 항목(rec.title)이 자동으로 그룹 기준이 된다(SQL의 GROUP BY를 따로 쓰지 않는다).
rank_query = """
MATCH (m:Movie {title: $movie})<-[:RATED]-(u:User)-[:RATED]->(rec:Movie)
RETURN rec.title AS recommendation, count(u) AS users
ORDER BY users DESC
LIMIT 20
"""
for row in graph.query(rank_query, params={"movie": "Crimson Tide"}):
    print(row["users"], row["recommendation"])

70 Forrest Gump
68 Dances with Wolves
68 Pulp Fiction
65 Fugitive, The
64 True Lies
63 Jurassic Park
62 Silence of the Lambs, The
61 Apollo 13
61 Batman
58 Clear and Present Danger
58 Aladdin
58 Braveheart
57 Shawshank Redemption, The
56 Seven (a.k.a. Se7en)
54 Speed
53 Firm, The
53 Terminator 2: Judgment Day
53 Beauty and the Beast
52 Die Hard: With a Vengeance
51 Waterworld


#### 과제 2 해설

실행 결과 (2026-11-01): Forrest Gump(70명), Dances with Wolves(68명), Pulp Fiction(68명), Fugitive, The(65명), True Lies(64명) 순입니다.

- `RETURN rec.title, count(u)`에서 `count(u)`가 `rec.title` 기준으로 자동으로 묶입니다. SQL의 `GROUP BY`를 따로 쓰지 않습니다(실습 가이드 1-4). 집계하면 같은 영화가 한 행으로 모이므로 `DISTINCT`도 필요 없습니다.
- 상위가 모두 많은 사람이 본 인기작입니다. 함께 평가한 사람 수만으로 고르면 Crimson Tide와 상관없이 인기작이 올라오는 인기 편향이 생깁니다. 실제 추천에서는 전체 평가 수로 나누거나, 높은 별점만 세는 식으로 보정합니다.

### 과제 3. Drama 장르의 평점 높은 영화

Drama 장르이고 imdbRating이 8.5 이상인 영화를 평점이 높은 순서로 조회하세요.

In [10]:
# 과제 3 정답
# 장르는 Movie의 속성이 아니라 Genre 노드다. IN_GENRE 관계로 이어진 노드를 이름으로 찾는다.
rows = graph.query("""
MATCH (m:Movie)-[:IN_GENRE]->(:Genre {name: "Drama"})
WHERE m.imdbRating >= 8.5
RETURN m.title AS title, m.imdbRating AS rating
ORDER BY rating DESC
""")
print(len(rows), "편")
# 출력은 상위 10편만 보여 준다.
for row in rows[:10]:
    print(row["rating"], row["title"])

64 편
9.6 Band of Brothers
9.3 Shawshank Redemption, The
9.2 Godfather, The
9.2 Decalogue, The (Dekalog)
9.1 Pride and Prejudice
9.0 Godfather: Part II, The
9.0 Dark Knight, The
8.9 12 Angry Men
8.9 Lord of the Rings: The Return of the King, The
8.9 Schindler's List


#### 과제 3 해설

실행 결과 (2026-11-01): 64편. 상위는 Band of Brothers(9.6), Shawshank Redemption, The(9.3), Godfather, The(9.2), Decalogue, The (Dekalog)(9.2), Pride and Prejudice(9.1)입니다.

- 장르 조건은 속성이 아니라 관계 패턴 `(m)-[:IN_GENRE]->(:Genre {name: "Drama"})`으로 겁니다. 장르가 노드로 저장되어 있기 때문입니다.
- imdbRating이 없는 영화는 `WHERE`에서 자동으로 빠집니다. `null >= 8.5`는 참이 아니기 때문입니다.
- Band of Brothers, Decalogue, Pride and Prejudice는 TV 시리즈인데 Movie 노드로 들어 있습니다. 데이터셋을 쓸 때는 레이블 이름과 실제 내용이 다를 수 있다는 점을 확인합니다.